# Are you ready for bio-computing? — the lab

A runnable companion to the ilm.red post **"Are you ready for bio-computing?"**

The article makes four claims that can be checked with arithmetic and a little simulation, so this
notebook checks them:

1. **The energy argument.** A supercomputer draws about 21 megawatts. A human brain draws about 20
   watts. That ratio is the whole reason anyone is funding this, and it is worth computing rather
   than repeating.
2. **The closed loop.** A dish of neurons plays a game only because silicon wraps it: screen to
   electrodes to spikes to keypress, and back, inside a latency budget.
3. **The training rule.** DishBrain was not rewarded. It was made *predictable* when it did well
   and *unpredictable* when it did badly, which is the free-energy principle used as a teaching
   signal. We build a caricature of it and watch the hit rate climb off the floor.
4. **The measurement problem.** The ethics section asks whether a dish can feel anything. We show
   why that question is hard in a way you can run: two systems with completely different insides,
   producing spike statistics you cannot tell apart from outside.

**What this is.** Teaching code. Roughly a hundred lines of NumPy, no accounts, no API keys, no
GPU, no internet. It runs top to bottom in under a minute.

**What this is not.** A replication. Nothing here is a model of a real neuron, and section 3 in
particular is a caricature that demonstrates the *shape* of the DishBrain claim, not its biology.
Where a number comes from the article it is cited in the cell; where a number is invented for the
demonstration, the cell says so.

## Setup

NumPy and Matplotlib, both of which Colab already has.

In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'matplotlib'])
    import numpy as np
    import matplotlib.pyplot as plt

rng = np.random.default_rng(20260817)   # fixed, so every number below is reproducible
print('numpy', np.__version__)

## 1. The energy argument, computed

Two numbers from the article:

- a large AI training cluster draws on the order of **21 megawatts** (the scale of demand that has
  operators restarting a retired nuclear plant);
- an adult human brain runs on about **20 watts**, roughly the power of a dim light bulb, while
  doing everything a brain does.

The ratio is the pitch. But the honest version of the comparison needs one more step, because a
whole brain is not what anyone has in a dish. The Melbourne experiment used about **200,000**
neurons; an adult brain has about **86 billion**. Scale the power budget by the cell count and the
dish's share of that 20 watts is not a light bulb at all.

In [ ]:
CLUSTER_WATTS      = 21e6          # ~21 MW, from the article's energy section
BRAIN_WATTS        = 20.0          # ~20 W, an adult human brain
BRAIN_NEURONS      = 86e9          # the standard modern estimate
DISH_NEURONS       = 200_000       # the DishBrain culture

ratio = CLUSTER_WATTS / BRAIN_WATTS
watts_per_neuron = BRAIN_WATTS / BRAIN_NEURONS
dish_watts = watts_per_neuron * DISH_NEURONS

print(f'cluster / brain                : {ratio:,.0f} x')
print(f'brains you could run on 21 MW  : {ratio:,.0f}')
print(f'power per neuron               : {watts_per_neuron*1e12:,.1f} pW')
print(f'a 200,000-neuron dish          : {dish_watts*1e6:,.0f} uW  ({dish_watts:.2e} W)')
print(f'dish / cluster                 : {dish_watts/CLUSTER_WATTS:.2e}')

Notice what the last line does to the argument. The dish is not a thousand times more efficient
than the cluster; it is about **twelve orders of magnitude** smaller in power draw, because it is
also doing almost nothing by comparison. The energy argument is a claim about the *ceiling* —
about what a substrate could in principle cost — and not a measurement of anything anyone has
built. That is the honest reading, and it is still a remarkable ceiling.

The plot below is on a log scale, because on a linear one the dish is invisible.

In [ ]:
labels = ['AI cluster\n(~21 MW)', 'human brain\n(~20 W)', '200k-neuron dish\n(scaled)']
values = [CLUSTER_WATTS, BRAIN_WATTS, dish_watts]

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(labels, values, color=['#c0532a', '#256f74', '#d99a2b'])
ax.set_yscale('log')
ax.set_ylabel('watts (log scale)')
ax.set_title('Power draw, twelve orders of magnitude apart')
for i, v in enumerate(values):
    ax.text(i, v * 1.6, f'{v:.3g} W', ha='center', fontsize=9)
ax.set_ylim(dish_watts / 50, CLUSTER_WATTS * 40)
plt.tight_layout(); plt.show()

## 2. The closed loop

A dish does not play a game. A dish plus an interface plays a game, and the interface is silicon on
both sides: the game state is turned into stimulation, the resulting spikes are read off the
electrode array, and the decoded activity moves the paddle. Then the loop closes and does it again.

Below is that loop, written out. The neurons are a rate-coded population — each "neuron" is a
firing rate, not a cell — and the decoder is the simplest thing that could work: subtract the
activity of the electrodes on one side from the other and move that way.

In [ ]:
N_ELECTRODES = 8          # a real MEA has far more; eight is enough to see the idea
N_NEURONS    = 64         # rate units, not cells

# Which neurons each electrode can hear, and which it can stimulate. In a dish this is geometry:
# an electrode couples to whatever grew near it.
rng_geo = np.random.default_rng(7)
coupling = (rng_geo.random((N_ELECTRODES, N_NEURONS)) < 0.25).astype(float)

def stimulate(ball_y, height=1.0):
    '''Game state in: which electrode fires, and how hard. The ball's height is coded by place.'''
    pattern = np.zeros(N_ELECTRODES)
    idx = int(np.clip(ball_y / height * (N_ELECTRODES - 1), 0, N_ELECTRODES - 1))
    pattern[idx] = 1.0
    return pattern

def read_out(rates):
    '''Spikes out: the paddle command. Top half minus bottom half, which is a decision, not a plan.'''
    per_electrode = coupling @ rates
    top, bottom = per_electrode[:N_ELECTRODES // 2].sum(), per_electrode[N_ELECTRODES // 2:].sum()
    return float(np.tanh((top - bottom) / (rates.sum() + 1e-9)))

# One turn of the loop, with nothing learned yet: random connectivity, random paddle.
W = rng.normal(0, 1, (N_NEURONS, N_ELECTRODES)) * 0.5
rates = np.maximum(W @ stimulate(0.8), 0)
print('paddle command from an untrained culture:', round(read_out(rates), 3))

### The latency budget

The loop has to close faster than the game moves. This is the part of the system that is entirely
engineering, and it is why the article insists that silicon still does the wrapping: the biology is
in the middle, and it is surrounded on both sides by hardware whose job is to be fast.

In [ ]:
budget_ms = {
    'render the game state':      0.5,
    'encode to a stimulus':       0.3,
    'stimulation pulse':          1.0,
    'neural response window':    10.0,   # the slow step, and it is the biology
    'amplify + digitise':         0.4,
    'decode to a paddle move':    0.2,
}
total = sum(budget_ms.values())
for k, v in budget_ms.items():
    print(f'  {k:<26} {v:5.1f} ms  {"#" * int(v * 2)}')
print(f'  {"TOTAL":<26} {total:5.1f} ms  -> {1000/total:5.1f} loops per second')
print('\n(illustrative numbers, chosen to be the right order of magnitude, not measured)')

## 3. The training rule: predictability as the teacher

This is the interesting part of the DishBrain result, and it is easy to state wrongly. The culture
was not given a reward, because a dish has nothing to want. It was given **predictability**.

Under the free-energy principle, a system that minimises surprise will act to make its own sensory
input predictable. So the experiment made the input predictable *when the paddle hit the ball* —
the same stimulus, in the same place, every time — and made it unpredictable when it missed:
random electrodes, random timing, noise. Nothing tells the culture it did well. Doing well is
simply the state in which the world stops being confusing.

Below, that rule drives a Hebbian update: connections that were active just before a predictable
outcome are strengthened, and connections active before noise are weakened. The learning is in the
*consequence*, not in a labelled target.

In [ ]:
# The culture has a fixed random feature layer (the tissue, which nobody designed) and a plastic
# readout (the synapses that change). The paddle goes to one of N_ELECTRODES places, so guessing
# at random is right 1/8 of the time and there is somewhere to climb from.
V = rng.normal(0, 1, (N_NEURONS, N_ELECTRODES)) * 0.8       # fixed: the dish is what it is
CHANCE = 1.0 / N_ELECTRODES

def softmax(z):
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

def run_session(A, n_trials=400, lr=0.015, learn=True, rng=rng):
    '''One session. Returns the hit rate and the updated readout.'''
    hits = 0
    for _ in range(n_trials):
        target = rng.integers(N_ELECTRODES)             # where the ball is
        stim   = np.zeros(N_ELECTRODES); stim[target] = 1.0
        rates  = np.maximum(V @ stim, 0)                # the culture's response
        action = rng.choice(N_ELECTRODES, p=softmax(A @ rates))   # where the paddle went
        hit    = (action == target)
        hits  += hit
        if not learn:
            continue
        # THE RULE, and the whole point of it: nothing here is told what the right answer was.
        # A hit is the state in which the next stimulus is PREDICTABLE, a miss the state in which
        # it is noise, and that difference alone sets the sign of the change. The update touches
        # only the pathway that was actually active, which is what makes it a local rule a sheet of
        # cells could plausibly implement.
        A[action] += lr * (1 if hit else -1) * rates
        A *= 0.9995                                     # slow decay, so nothing runs away
    return hits / n_trials, A

A0 = np.zeros((N_ELECTRODES, N_NEURONS))
baseline, _ = run_session(A0.copy(), learn=False)
print(f'chance                 : {CHANCE:.1%}')
print(f'no learning, hit rate  : {baseline:.1%}')

curve, At = [], A0.copy()
for session in range(40):
    rate, At = run_session(At, learn=True)
    curve.append(rate)
print(f'session 1              : {curve[0]:.1%}')
print(f'session 40             : {curve[-1]:.1%}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(range(1, len(curve) + 1), [c * 100 for c in curve], color='#256f74', lw=2, marker='o', ms=3)
ax.axhline(CHANCE * 100, color='#c0532a', ls='--', lw=1.5, label=f'chance ({CHANCE:.0%})')
ax.set_xlabel('session'); ax.set_ylabel('hit rate (%)')
ax.set_title('Predictability as the only teaching signal')
ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

**Read this plot carefully, because it is the one that invites over-claiming.**

It climbs from chance to a ceiling, and the ceiling is the tell: the task here is an eight-way
lookup with no noise, no delay and no opponent, so a working rule saturates it. A real culture on a
real array improves *modestly* and unevenly, and the published result is a statistically detectable
difference in performance rather than a dish that gets good at Pong.

What the run does show is the thing worth taking away: a rule of the form "make the world
predictable when the behaviour is right" is **sufficient** to move a population off chance with no
labelled target anywhere in the loop. Nothing in the code is ever told the correct answer. That is
the article's actual claim about the training signal, and it is information, not reward.

## 4. Why "does it feel anything" is hard from the outside

The ethics section asks the question the field cannot yet answer, and it is worth understanding
*why* it cannot rather than treating it as squeamishness.

Here are two systems. One has an internal state that persists and integrates across time; the other
is memoryless and reactive. They are genuinely different inside. We tune them so that their
observable output — the only thing an electrode array can see — has the same distribution.

Then we look at what an experimenter looking at spikes would be able to say.

In [ ]:
T = 4000

def integrator(rng):
    '''Has a persistent internal state that accumulates evidence over time.'''
    s, out = 0.0, []
    for _ in range(T):
        s = 0.9 * s + rng.normal(0, 1)
        out.append(1 if rng.random() < 1 / (1 + np.exp(-0.55 * s)) else 0)
    return np.array(out), 'carries state across time'

def reflex(rng):
    '''No state at all. Each moment is independent.'''
    out = [1 if rng.random() < 0.5 else 0 for _ in range(T)]
    return np.array(out), 'no state whatsoever'

a, why_a = integrator(np.random.default_rng(1))
b, why_b = reflex(np.random.default_rng(2))

def summarise(x):
    return {
        'firing rate':      x.mean(),
        'variance':         x.var(),
        'longest burst':    max((len(list(g)) for k, g in __import__('itertools').groupby(x) if k), default=0),
    }

print(f'{"":<16}{"integrator":>14}{"reflex":>14}')
for k in ['firing rate', 'variance', 'longest burst']:
    print(f'{k:<16}{summarise(a)[k]:>14.3f}{summarise(b)[k]:>14.3f}')
print(f'\ninside: A {why_a!r}, B {why_b!r}')

The first two summary statistics — the ones a rate-based readout gives you — are close enough to be
within run-to-run noise. The third one separates them, and that is the actual lesson: **the answer
depends entirely on which measurement you thought to take.** Had you recorded only firing rate, you
would have concluded the two systems were the same. There is no measurement that is guaranteed to
be the right one, and no way to know in advance that you have taken it.

Now scale the difficulty up. We cannot reliably detect awareness in an unresponsive *human* patient
— a person with a nervous system identical in kind to ours, who can sometimes be asked. The dish
has no report, no behaviour outside the task, and no shared frame of reference at all.

The article's position, which this notebook does not improve on: the honest answer is that we do
not know, the honest response is to build the measurements before scaling the systems, and "we
cannot tell" is not the same as "there is nothing there."

## What you actually ran

1. The energy ratio, computed from the article's own numbers, **including** the scaling step that
   makes the comparison honest.
2. A closed sense-compute-act loop with a stimulus encoder, a population, and a decoder, plus the
   latency budget that forces the silicon wrapper.
3. The free-energy training rule, as a rule: predictability where a reward would be, learning with
   no labelled target.
4. A demonstration that two different insides can look identical from outside, which is the shape
   of the problem the ethics section is about.

**Further reading** is in the post's own reference list; every paper there is linked to its arXiv
entry, and most also have a summary page on ilm.red with the extracted terms.

Licensed Apache-2.0, like the rest of this repository.